# Jev-style VQA 로컬 실험 — RTX 5060 Ti 16GB (환경 복구 v2)

Qwen2.5-VL-3B 기반으로 `generate / letter scoring / choice-text scoring / ensemble`을 비교합니다.

**Windows + Python 3.12 + RTX 5060 Ti 16GB 사용 가능**하도록 수정했습니다. 기존 환경에 남아 있는 깨진 `torchaudio` 때문에 `transformers` import가 실패하는 문제를 자동으로 제거합니다.

> 처음에는 `LIMIT = 1`로 smoke test → 성공 후 `200~300`으로 늘리세요.


## 0. 환경 복구 셀 — 오류가 있거나 최초 설치할 때만 실행

현재 커널의 Python에 직접 설치합니다. **이 셀 실행이 끝나면 반드시 커널을 재시작한 뒤, 이 셀은 건너뛰고 `1. 환경 확인`부터 실행하세요.**

이번 VQA 실험에는 오디오가 필요 없으므로 `torchaudio`는 설치하지 않습니다. 기존 환경에 깨진 `torchaudio`가 남아 있으면 Transformers가 로딩 과정에서 실패할 수 있습니다.


In [1]:
# 최초 설치 / 현재 오류 복구용 셀
# 실행 완료 후 반드시 VS Code/Jupyter의 "Restart Kernel"을 누르세요.
import sys
import subprocess
import importlib.metadata as md

PY = sys.executable
print("현재 커널 Python:", PY)
print("Python version   :", sys.version.split()[0])

def installed_version(name):
    try:
        return md.version(name)
    except md.PackageNotFoundError:
        return None

print("\n[설치 전]")
for pkg in ["torch", "torchvision", "torchaudio", "transformers"]:
    print(f"{pkg:12s}:", installed_version(pkg))

# 1) 문제를 일으킨 torchaudio를 먼저 제거합니다.
#    이 실험은 이미지+텍스트 VQA라 torchaudio가 필요 없습니다.
subprocess.run(
    [PY, "-m", "pip", "uninstall", "-y", "torchaudio"],
    check=False
)

# 2) torch / torchvision 버전이 정확하지 않을 때만 동일 CUDA wheel 조합으로 재설치
torch_v = installed_version("torch") or ""
tv_v = installed_version("torchvision") or ""

need_torch_repair = not (
    torch_v.startswith("2.12.1") and
    tv_v.startswith("0.27.1")
)

if need_torch_repair:
    print("\n[PyTorch 조합 복구]")
    subprocess.run(
        [PY, "-m", "pip", "uninstall", "-y", "torch", "torchvision", "torchaudio"],
        check=False
    )
    subprocess.check_call([
        PY, "-m", "pip", "install",
        "--no-cache-dir",
        "torch==2.12.1",
        "torchvision==0.27.1",
        "--index-url", "https://download.pytorch.org/whl/cu130"
    ])
else:
    print("\nPyTorch / torchvision 버전은 이미 목표 조합입니다. 재설치를 건너뜁니다.")

# 3) 나머지 의존성
subprocess.check_call([PY, "-m", "pip", "install", "--upgrade", "pip"])
subprocess.check_call([
    PY, "-m", "pip", "install", "--upgrade",
    "transformers==5.17.0",
    "huggingface-hub>=0.35,<2",
    "safetensors>=0.5",
    "pillow>=11,<13",
    "pandas>=2.2,<3.1",
    "numpy>=1.26,<3",
    "tqdm>=4.66"
])

print("\n[설치 후 - 패키지 메타데이터]")
for pkg in ["torch", "torchvision", "torchaudio", "transformers"]:
    print(f"{pkg:12s}:", installed_version(pkg))

print("\n" + "=" * 72)
print("환경 복구가 끝났습니다.")
print("중요: 지금 바로 VS Code/Jupyter에서 'Restart Kernel'을 실행하세요.")
print("재시작 후 이 0번 셀은 다시 실행하지 말고, 1번 환경 확인 셀부터 실행하세요.")
print("=" * 72)


현재 커널 Python: c:\SSAFY\AIChallenge\baseline\Scripts\python.exe
Python version   : 3.12.10

[설치 전]
torch       : 2.12.1+cu130
torchvision : 0.27.1+cu130
torchaudio  : 2.11.0+cu128
transformers: 5.17.0

PyTorch / torchvision 버전은 이미 목표 조합입니다. 재설치를 건너뜁니다.

[설치 후 - 패키지 메타데이터]
torch       : 2.12.1+cu130
torchvision : 0.27.1+cu130
torchaudio  : None
transformers: 5.17.0

환경 복구가 끝났습니다.
중요: 지금 바로 VS Code/Jupyter에서 'Restart Kernel'을 실행하세요.
재시작 후 이 0번 셀은 다시 실행하지 말고, 1번 환경 확인 셀부터 실행하세요.


## 1. 환경 확인

0번 셀을 실행했다면 **커널 재시작 후** 아래 셀을 실행합니다. `torchaudio`는 `NOT INSTALLED`가 정상입니다.


In [1]:
import sys
import importlib.metadata as md

def pkg_version(name):
    try:
        return md.version(name)
    except md.PackageNotFoundError:
        return "NOT INSTALLED"

print("Kernel Python:", sys.executable)
print("Python       :", sys.version.split()[0])
print("torchaudio   :", pkg_version("torchaudio"), "  <-- NOT INSTALLED가 정상")

# 중요: 위 확인 뒤에 실제 binary 패키지를 import
import torch
import torchvision
import transformers

print("torch        :", torch.__version__)
print("torchvision  :", torchvision.__version__)
print("transformers :", transformers.__version__)
print("CUDA runtime :", torch.version.cuda)
print("CUDA avail   :", torch.cuda.is_available())

if not torch.__version__.startswith("2.12.1"):
    raise RuntimeError(f"torch 버전이 예상과 다릅니다: {torch.__version__}")
if not torchvision.__version__.startswith("0.27.1"):
    raise RuntimeError(f"torchvision 버전이 예상과 다릅니다: {torchvision.__version__}")

# torchvision native op가 정상 연결되는지 확인
try:
    _ = torch.ops.torchvision.nms
    print("torchvision::nms: OK")
except Exception as e:
    raise RuntimeError(
        "torch/torchvision binary 조합이 아직 깨져 있습니다. "
        "0번 환경 복구 셀을 실행 → 커널 재시작 후 다시 확인하세요."
    ) from e

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA를 사용할 수 없습니다. NVIDIA 드라이버와 현재 VS Code 커널을 확인하세요."
    )

print("GPU          :", torch.cuda.get_device_name(0))
print("VRAM(GB)     :", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
print("BF16 support :", torch.cuda.is_bf16_supported())

# Transformers VLM import까지 여기서 조기 검증
from transformers import AutoModelForImageTextToText, AutoProcessor
print("Transformers VLM import: OK")


Kernel Python: c:\SSAFY\AIChallenge\baseline\Scripts\python.exe
Python       : 3.12.10
torchaudio   : NOT INSTALLED   <-- NOT INSTALLED가 정상


c:\SSAFY\AIChallenge\baseline\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


torch        : 2.12.1+cu130
torchvision  : 0.27.1+cu130
transformers : 5.17.0
CUDA runtime : 13.0
CUDA avail   : True
torchvision::nms: OK
GPU          : NVIDIA GeForce RTX 5060 Ti
VRAM(GB)     : 15.93
BF16 support : True
Transformers VLM import: OK


## 2. 실험 설정
`DATA_DIR`만 자신의 데이터셋 경로로 수정하면 됩니다.

In [13]:
DATA_DIR = r"C:\SSAFY\AIChallenge\dataset"
MODE = "val"                 # "val" 또는 "test"
LIMIT = 300                    # smoke test=1, 빠른 비교=200~300, 전체=0
METHODS = "generate,letter,text,ensemble" # 이후 "generate,letter,text,ensemble"
SUBMIT_SCORER = "letter"    # test일 때 generate/letter/text/ensemble
MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"
OUTPUT_DIR = "outputs_jev"
CACHE_DIR = ".hf_cache"
SEED = 42
MAX_VISUAL_TOKENS = 768
LOW_MAX_VISUAL_TOKENS = 384
MAX_NEW_TOKENS = 8
OFFLINE = False  # 모델 캐시가 완료된 뒤 True로 설정 가능


## 3. Jev-style VQA 엔진
아래 셀은 데이터 스키마/이미지 경로 자동 탐색, Qwen2.5-VL 로딩, letter/text likelihood, OOM 저해상도 재시도, 제출 파일 생성을 포함합니다.

In [14]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
Local Jev-style multiple-choice VQA experiment for Windows + RTX 5060 Ti 16GB.

Default model: Qwen/Qwen2.5-VL-3B-Instruct
No bitsandbytes / flash-attn / qwen-vl-utils required.

Examples
--------
# 1) environment/model smoke test with one validation sample
python jev_vqa_local.py --data-dir "C:\\SSAFY\\AIChallenge\\dataset" --mode val --limit 1 --methods generate,letter

# 2) quick 200-sample comparison
python jev_vqa_local.py --data-dir "C:\\SSAFY\\AIChallenge\\dataset" --mode val --limit 200 --methods generate,letter,text,ensemble

# 3) test inference + submission
python jev_vqa_local.py --data-dir "C:\\SSAFY\\AIChallenge\\dataset" --mode test --methods letter --submit-scorer letter

# 4) after model is already cached, force offline mode
python jev_vqa_local.py --data-dir "C:\\SSAFY\\AIChallenge\\dataset" --mode test --methods letter --offline
"""

from __future__ import annotations

import argparse
import gc
import json
import math
import os
import random
import re
import subprocess
import sys
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image, ImageOps
from tqdm import tqdm

try:
    import torchvision
except Exception as e:
    raise RuntimeError(
        "torchvision import failed. Run notebook Cell 0 (environment repair), restart the kernel, "
        "then retry. Original error: " + repr(e)
    ) from e

try:
    import transformers
    from transformers import AutoModelForImageTextToText, AutoProcessor
except Exception as e:
    raise RuntimeError(
        "transformers import failed. Run notebook Cell 0 (environment repair), restart the kernel, "
        "then retry. A stale/broken torchaudio installation is a common cause. Original error: " + repr(e)
    ) from e


LETTERS = ["A", "B", "C", "D"]
IMAGE_EXTS = [".jpg", ".jpeg", ".png", ".webp", ".bmp"]


@dataclass
class Schema:
    id_col: str
    image_col: str
    question_col: str
    option_cols: List[str]
    label_col: Optional[str]


def parse_args() -> argparse.Namespace:
    p = argparse.ArgumentParser(formatter_class=argparse.ArgumentDefaultsHelpFormatter)
    p.add_argument("--data-dir", type=str, required=True)
    p.add_argument("--mode", choices=["val", "test"], default="val")
    p.add_argument("--csv", type=str, default=None, help="Optional explicit CSV path")
    p.add_argument("--model-id", type=str, default="Qwen/Qwen2.5-VL-3B-Instruct")
    p.add_argument("--cache-dir", type=str, default=".hf_cache")
    p.add_argument("--output-dir", type=str, default="outputs_jev")
    p.add_argument("--methods", type=str, default="generate,letter", help="generate,letter,text,ensemble")
    p.add_argument("--submit-scorer", choices=["generate", "letter", "text", "ensemble"], default="letter")
    p.add_argument("--limit", type=int, default=0, help="0 = all rows")
    p.add_argument("--seed", type=int, default=42)
    p.add_argument("--max-visual-tokens", type=int, default=768, help="Qwen pixels = tokens * 28 * 28")
    p.add_argument("--low-max-visual-tokens", type=int, default=384, help="OOM fallback")
    p.add_argument("--max-new-tokens", type=int, default=8)
    p.add_argument("--offline", action="store_true", help="Use only cached HF files")
    p.add_argument("--trust-remote-code", action="store_true", help="Normally not needed for Qwen2.5-VL")
    return p.parse_args()


def print_environment() -> None:
    print("=" * 80)
    print("Environment")
    print(f"Python       : {sys.version.split()[0]}")
    print(f"PyTorch      : {torch.__version__}")
    print(f"TorchVision  : {torchvision.__version__}")
    print(f"Transformers : {transformers.__version__}")
    print(f"CUDA runtime : {torch.version.cuda}")
    print(f"CUDA avail   : {torch.cuda.is_available()}")

    if torch.cuda.is_available():
        props = torch.cuda.get_device_properties(0)
        print(f"GPU          : {props.name}")
        print(f"VRAM         : {props.total_memory / 1024**3:.2f} GB")
        print(f"Compute cap  : {props.major}.{props.minor}")
        try:
            print(f"BF16 support : {torch.cuda.is_bf16_supported()}")
        except Exception:
            pass
    try:
        smi = subprocess.run(
            ["nvidia-smi", "--query-gpu=driver_version,name,memory.total", "--format=csv,noheader"],
            capture_output=True,
            text=True,
            timeout=5,
            check=False,
        )
        if smi.stdout.strip():
            print(f"nvidia-smi   : {smi.stdout.strip()}")
    except Exception:
        pass
    print("=" * 80)


def validate_cuda() -> torch.dtype:
    if not torch.cuda.is_available():
        raise RuntimeError(
            "CUDA GPU is not available. Run notebook Cell 0, restart the VS Code/Jupyter kernel, "
            "and verify that the selected kernel is the intended local environment."
        )
    name = torch.cuda.get_device_name(0)
    if "5060" not in name:
        print(f"[WARN] Expected RTX 5060-class GPU, detected: {name}")

    # Blackwell supports BF16; keep FP16 fallback for portability.
    try:
        return torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    except Exception:
        return torch.float16


def first_existing(columns: Sequence[str], candidates: Sequence[str]) -> Optional[str]:
    by_lower = {str(c).lower(): str(c) for c in columns}
    for c in candidates:
        if c.lower() in by_lower:
            return by_lower[c.lower()]
    return None


def detect_option_cols(df: pd.DataFrame) -> List[str]:
    cols = [str(c) for c in df.columns]
    lower = {c.lower(): c for c in cols}

    patterns = [
        ["A", "B", "C", "D"],
        ["option_a", "option_b", "option_c", "option_d"],
        ["choice_a", "choice_b", "choice_c", "choice_d"],
        ["answer_a", "answer_b", "answer_c", "answer_d"],
        ["option1", "option2", "option3", "option4"],
        ["option_1", "option_2", "option_3", "option_4"],
        ["choice1", "choice2", "choice3", "choice4"],
        ["choice_1", "choice_2", "choice_3", "choice_4"],
    ]
    for pattern in patterns:
        if all(x.lower() in lower for x in pattern):
            return [lower[x.lower()] for x in pattern]

    raise ValueError(
        "Could not detect four answer-option columns. Supported examples: A/B/C/D, "
        "option_a..option_d, choice1..choice4.\n"
        f"CSV columns: {cols}"
    )


def detect_schema(df: pd.DataFrame, mode: str) -> Schema:
    cols = [str(c) for c in df.columns]
    id_col = first_existing(cols, ["id", "ID", "sample_id", "uid", "index"])
    question_col = first_existing(cols, ["question", "Question", "query", "prompt", "text"])
    image_col = first_existing(
        cols,
        ["image", "image_path", "img_path", "img", "filename", "file_name", "image_id", "id", "ID"],
    )
    label_col = first_existing(cols, ["answer", "label", "target", "correct_answer", "correct", "gt"])
    option_cols = detect_option_cols(df)

    if id_col is None:
        # Stable synthetic ID if dataset has none.
        df["__row_id__"] = np.arange(len(df))
        id_col = "__row_id__"
    if question_col is None:
        raise ValueError(f"Could not detect question column. CSV columns: {cols}")
    if image_col is None:
        image_col = id_col
    if mode == "val" and label_col is None:
        raise ValueError(
            "Validation mode needs a label column. Supported names include answer/label/target/correct_answer."
        )

    return Schema(id_col=id_col, image_col=image_col, question_col=question_col, option_cols=option_cols, label_col=label_col)


def resolve_image_path(data_dir: Path, raw_value: object, mode: str) -> Path:
    raw = str(raw_value).strip()
    p = Path(raw)
    candidates: List[Path] = []

    if p.is_absolute():
        candidates.append(p)
    else:
        common_dirs = [
            data_dir,
            data_dir / "images",
            data_dir / f"{mode}_images",
            data_dir / ("train_images" if mode == "val" else "test_images"),
            data_dir / ("train" if mode == "val" else "test"),
        ]
        for d in common_dirs:
            candidates.append(d / p)

    expanded: List[Path] = []
    for c in candidates:
        expanded.append(c)
        if c.suffix == "":
            expanded.extend(c.with_suffix(ext) for ext in IMAGE_EXTS)

    for c in expanded:
        if c.exists() and c.is_file():
            return c.resolve()

    # Slower fallback: exact stem search inside common image folders.
    stem = p.stem
    search_dirs = [
        data_dir,
        data_dir / "images",
        data_dir / "train_images",
        data_dir / "test_images",
        data_dir / "train",
        data_dir / "test",
    ]
    for d in search_dirs:
        if d.exists():
            for ext in IMAGE_EXTS:
                hit = d / f"{stem}{ext}"
                if hit.exists():
                    return hit.resolve()

    preview = "\n".join(f"  - {x}" for x in expanded[:12])
    raise FileNotFoundError(f"Image not found for value={raw!r}. Tried:\n{preview}")


def infer_numeric_label_base(series: pd.Series) -> Optional[int]:
    vals = []
    for x in series.dropna().tolist():
        sx = str(x).strip()
        if re.fullmatch(r"-?\d+", sx):
            vals.append(int(sx))
        else:
            return None
    if not vals:
        return None
    u = set(vals)
    if u.issubset({0, 1, 2, 3}) and 0 in u:
        return 0
    if u.issubset({1, 2, 3, 4}) and 4 in u:
        return 1
    if u.issubset({0, 1, 2, 3}):
        return 0
    if u.issubset({1, 2, 3, 4}):
        return 1
    return None


def normalize_label(value: object, options: Sequence[str], numeric_base: Optional[int] = None) -> Optional[str]:
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return None
    s = str(value).strip()
    su = s.upper()
    if su in LETTERS:
        return su

    # Numeric labels: dataset-level base is inferred to avoid 1/2/3 ambiguity.
    if re.fullmatch(r"-?\d+", s):
        n = int(s)
        if numeric_base == 0 and 0 <= n <= 3:
            return LETTERS[n]
        if numeric_base == 1 and 1 <= n <= 4:
            return LETTERS[n - 1]

    # Label may be the option text itself.
    for i, opt in enumerate(options):
        if s == str(opt).strip():
            return LETTERS[i]
    return None


def parse_generated_letter(text: str) -> Optional[str]:
    t = text.strip().upper()
    # Prefer standalone A-D near the beginning.
    m = re.search(r"(?:^|\b)([ABCD])(?:\b|$)", t)
    if m:
        return m.group(1)
    # Fallback common forms: "(B)", "B.", "ANSWER:B"
    m = re.search(r"[\(\[:\s]([ABCD])[\)\].,\s]", " " + t + " ")
    return m.group(1) if m else None


def softmax_np(scores: Sequence[float]) -> np.ndarray:
    x = np.asarray(scores, dtype=np.float64)
    x = x - np.nanmax(x)
    ex = np.exp(x)
    return ex / ex.sum()


def conf_margin(probs: Sequence[float]) -> Tuple[float, float]:
    x = sorted([float(v) for v in probs], reverse=True)
    return x[0], x[0] - x[1]


def build_letter_prompt(question: str, options: Sequence[str]) -> str:
    return (
        "You are solving a multiple-choice visual question. Inspect the image carefully.\n"
        f"Question: {question}\n\n"
        f"A. {options[0]}\n"
        f"B. {options[1]}\n"
        f"C. {options[2]}\n"
        f"D. {options[3]}\n\n"
        "Return exactly one letter only: A, B, C, or D."
    )


def build_text_prompt(question: str, options: Sequence[str]) -> str:
    return (
        "You are solving a multiple-choice visual question. Inspect the image carefully.\n"
        f"Question: {question}\n\n"
        f"A. {options[0]}\n"
        f"B. {options[1]}\n"
        f"C. {options[2]}\n"
        f"D. {options[3]}\n\n"
        "Return exactly the full text of the single correct choice, with no explanation and no letter prefix."
    )


class VQAEngine:
    def __init__(
        self,
        model_id: str,
        cache_dir: str,
        dtype: torch.dtype,
        max_visual_tokens: int,
        low_max_visual_tokens: int,
        offline: bool,
        trust_remote_code: bool,
    ) -> None:
        self.model_id = model_id
        self.device = torch.device("cuda:0")
        self.dtype = dtype
        self.cache_dir = str(Path(cache_dir).resolve())
        self.offline = offline
        self.trust_remote_code = trust_remote_code

        min_pixels = 256 * 28 * 28
        max_pixels = max_visual_tokens * 28 * 28
        low_max_pixels = low_max_visual_tokens * 28 * 28

        print(f"Loading processor: {model_id}")
        self.processor = AutoProcessor.from_pretrained(
            model_id,
            cache_dir=self.cache_dir,
            local_files_only=offline,
            min_pixels=min_pixels,
            max_pixels=max_pixels,
            trust_remote_code=trust_remote_code,
        )
        self.processor_low = AutoProcessor.from_pretrained(
            model_id,
            cache_dir=self.cache_dir,
            local_files_only=offline,
            min_pixels=min_pixels,
            max_pixels=low_max_pixels,
            trust_remote_code=trust_remote_code,
        )

        print(f"Loading model: {model_id} ({dtype})")
        common = dict(
            cache_dir=self.cache_dir,
            local_files_only=offline,
            trust_remote_code=trust_remote_code,
            attn_implementation="sdpa",
        )
        # Transformers 5.x uses dtype broadly; older releases used torch_dtype.
        try:
            self.model = AutoModelForImageTextToText.from_pretrained(model_id, dtype=dtype, **common)
        except TypeError:
            self.model = AutoModelForImageTextToText.from_pretrained(model_id, torch_dtype=dtype, **common)

        self.model.to(self.device)
        self.model.eval()
        torch.set_grad_enabled(False)

        tokenizer = self.processor.tokenizer
        print("Candidate tokenization:")
        for x in LETTERS:
            ids = tokenizer(x, add_special_tokens=False)["input_ids"]
            print(f"  {x}: {ids}")

    def load_image(self, path: Path) -> Image.Image:
        with Image.open(path) as im:
            im = ImageOps.exif_transpose(im).convert("RGB")
            return im.copy()

    def prepare_base(self, image: Image.Image, prompt: str, low_res: bool = False):
        proc = self.processor_low if low_res else self.processor
        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image"},
                    {"type": "text", "text": prompt},
                ],
            }
        ]
        text = proc.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = proc(
            text=[text],
            images=[image],
            padding=True,
            return_tensors="pt",
        )
        return proc, text, inputs.to(self.device)

    @staticmethod
    def _copy_with_suffix(base_inputs, suffix_ids: torch.Tensor):
        base_len = base_inputs["input_ids"].shape[1]
        full = {}
        n = suffix_ids.shape[1]
        for k, v in base_inputs.items():
            if k == "input_ids":
                full[k] = torch.cat([v, suffix_ids], dim=1)
            elif k == "attention_mask":
                extra = torch.ones((v.shape[0], n), dtype=v.dtype, device=v.device)
                full[k] = torch.cat([v, extra], dim=1)
            elif torch.is_tensor(v) and v.ndim == 2 and v.shape[1] == base_len and "token_type" in k:
                # New answer tokens are text tokens.
                extra = torch.zeros((v.shape[0], n), dtype=v.dtype, device=v.device)
                full[k] = torch.cat([v, extra], dim=1)
            else:
                full[k] = v
        return full

    @torch.inference_mode()
    def score_suffix(self, proc, base_inputs, suffix: str) -> Tuple[float, float, int]:
        ids = proc.tokenizer(suffix, add_special_tokens=False, return_tensors="pt")["input_ids"].to(self.device)
        if ids.numel() == 0:
            return -1e9, -1e9, 0
        base_len = base_inputs["input_ids"].shape[1]
        full_inputs = self._copy_with_suffix(base_inputs, ids)
        out = self.model(**full_inputs, use_cache=False)
        # Candidate token j (at input index base_len+j) is predicted by logits at base_len+j-1.
        n = ids.shape[1]
        pred_logits = out.logits[:, base_len - 1 : base_len + n - 1, :].float()
        logp = F.log_softmax(pred_logits, dim=-1)
        token_lp = torch.gather(logp, dim=-1, index=ids.unsqueeze(-1)).squeeze(-1)
        total = float(token_lp.sum().item())
        mean = float(token_lp.mean().item())
        return total, mean, n

    @torch.inference_mode()
    def letter_scores(self, proc, base_inputs) -> Tuple[np.ndarray, np.ndarray]:
        # Fast path: one prompt forward if each A/B/C/D is exactly one token.
        token_lists = [proc.tokenizer(x, add_special_tokens=False)["input_ids"] for x in LETTERS]
        if all(len(x) == 1 for x in token_lists):
            out = self.model(**base_inputs, use_cache=False)
            last = out.logits[0, -1, :].float()
            raw = np.asarray([float(last[x[0]].item()) for x in token_lists], dtype=np.float64)
        else:
            raw = np.asarray([self.score_suffix(proc, base_inputs, x)[1] for x in LETTERS], dtype=np.float64)
        return raw, softmax_np(raw)

    @torch.inference_mode()
    def generate_letter(self, proc, base_inputs, max_new_tokens: int) -> Tuple[Optional[str], str]:
        input_len = base_inputs["input_ids"].shape[1]
        out = self.model.generate(
            **base_inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            use_cache=True,
        )
        new_ids = out[0, input_len:]
        text = proc.decode(new_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False).strip()
        return parse_generated_letter(text), text

    def infer_one(
        self,
        image_path: Path,
        question: str,
        options: Sequence[str],
        methods: set[str],
        max_new_tokens: int,
        low_res: bool = False,
    ) -> Dict[str, object]:
        image = self.load_image(image_path)
        result: Dict[str, object] = {}

        need_letter_base = bool(methods & {"generate", "letter", "ensemble"})
        need_text = bool(methods & {"text", "ensemble"})

        letter_probs = None
        text_probs = None

        if need_letter_base:
            proc, _, base = self.prepare_base(image, build_letter_prompt(question, options), low_res=low_res)

            if "generate" in methods:
                gpred, gtext = self.generate_letter(proc, base, max_new_tokens=max_new_tokens)
                result["generate_pred"] = gpred
                result["generate_text"] = gtext

            if methods & {"letter", "ensemble"}:
                raw, probs = self.letter_scores(proc, base)
                letter_probs = probs
                pred_idx = int(np.argmax(probs))
                conf, margin = conf_margin(probs)
                result["letter_pred"] = LETTERS[pred_idx]
                result["letter_confidence"] = conf
                result["letter_margin"] = margin
                for i, L in enumerate(LETTERS):
                    result[f"letter_score_{L}"] = float(raw[i])
                    result[f"letter_p_{L}"] = float(probs[i])

        if need_text:
            proc_t, _, base_t = self.prepare_base(image, build_text_prompt(question, options), low_res=low_res)
            # Mean token log-probability avoids automatically penalizing longer options.
            text_raw = np.asarray(
                [self.score_suffix(proc_t, base_t, str(opt))[1] for opt in options],
                dtype=np.float64,
            )
            text_probs = softmax_np(text_raw)
            pred_idx = int(np.argmax(text_probs))
            conf, margin = conf_margin(text_probs)
            result["text_pred"] = LETTERS[pred_idx]
            result["text_confidence"] = conf
            result["text_margin"] = margin
            for i, L in enumerate(LETTERS):
                result[f"text_score_{L}"] = float(text_raw[i])
                result[f"text_p_{L}"] = float(text_probs[i])

        if "ensemble" in methods:
            if letter_probs is None or text_probs is None:
                raise RuntimeError("ensemble requires both letter and text scoring")
            ens = 0.5 * letter_probs + 0.5 * text_probs
            pred_idx = int(np.argmax(ens))
            conf, margin = conf_margin(ens)
            result["ensemble_pred"] = LETTERS[pred_idx]
            result["ensemble_confidence"] = conf
            result["ensemble_margin"] = margin
            for i, L in enumerate(LETTERS):
                result[f"ensemble_p_{L}"] = float(ens[i])

        return result

    def infer_with_oom_fallback(self, *args, **kwargs) -> Tuple[Dict[str, object], bool]:
        try:
            return self.infer_one(*args, **kwargs, low_res=False), False
        except torch.OutOfMemoryError:
            print("\n[OOM] Retrying this sample at lower image resolution...")
            gc.collect()
            torch.cuda.empty_cache()
            return self.infer_one(*args, **kwargs, low_res=True), True
        except RuntimeError as e:
            if "out of memory" in str(e).lower():
                print("\n[OOM] Retrying this sample at lower image resolution...")
                gc.collect()
                torch.cuda.empty_cache()
                return self.infer_one(*args, **kwargs, low_res=True), True
            raise


def compute_summary(df: pd.DataFrame, methods: set[str]) -> Dict[str, object]:
    summary: Dict[str, object] = {"n": int(len(df))}
    if "true_label" not in df.columns:
        return summary

    for m in ["generate", "letter", "text", "ensemble"]:
        col = f"{m}_pred"
        if m in methods and col in df.columns:
            valid = df[col].notna() & df["true_label"].notna()
            if valid.any():
                summary[f"{m}_accuracy"] = float((df.loc[valid, col] == df.loc[valid, "true_label"]).mean())
                summary[f"{m}_valid_n"] = int(valid.sum())

    if "generate_pred" in df.columns and "letter_pred" in df.columns:
        valid = df["generate_pred"].notna() & df["letter_pred"].notna()
        if valid.any():
            summary["generate_letter_agreement"] = float(
                (df.loc[valid, "generate_pred"] == df.loc[valid, "letter_pred"]).mean()
            )
            if "true_label" in df.columns:
                g = df.loc[valid, "generate_pred"] == df.loc[valid, "true_label"]
                l = df.loc[valid, "letter_pred"] == df.loc[valid, "true_label"]
                summary["generate_wrong_letter_correct"] = int((~g & l).sum())
                summary["generate_correct_letter_wrong"] = int((g & ~l).sum())

    if "letter_pred" in df.columns and "text_pred" in df.columns:
        valid = df["letter_pred"].notna() & df["text_pred"].notna()
        if valid.any():
            summary["letter_text_agreement"] = float(
                (df.loc[valid, "letter_pred"] == df.loc[valid, "text_pred"]).mean()
            )
    return summary


def encode_predictions_for_sample(preds: pd.Series, sample_target: pd.Series) -> pd.Series:
    """Match common sample-submission target encodings when they are inferable."""
    nonnull = sample_target.dropna()
    if len(nonnull) == 0:
        return preds

    # Numeric placeholder/target: infer 0-based vs 1-based when possible.
    numeric = pd.to_numeric(nonnull, errors="coerce")
    if numeric.notna().all():
        vals = set(int(x) for x in numeric.tolist())
        if vals.issubset({0, 1, 2, 3}):
            return preds.map({"A": 0, "B": 1, "C": 2, "D": 3})
        if vals.issubset({1, 2, 3, 4}) and 4 in vals:
            return preds.map({"A": 1, "B": 2, "C": 3, "D": 4})

    upper = set(str(x).strip().upper() for x in nonnull.tolist())
    if upper.issubset(set(LETTERS)):
        return preds
    return preds


def make_submission(
    data_dir: Path,
    raw_df: pd.DataFrame,
    schema: Schema,
    result_df: pd.DataFrame,
    scorer: str,
    out_path: Path,
) -> None:
    pred_col = f"{scorer}_pred"
    if pred_col not in result_df.columns:
        raise ValueError(
            f"Submission scorer={scorer!r} was not run. Include it in --methods. "
            f"Available prediction columns: {[c for c in result_df.columns if c.endswith('_pred')]}"
        )
    if result_df[pred_col].isna().any():
        bad = int(result_df[pred_col].isna().sum())
        raise ValueError(f"{bad} rows have no {scorer} prediction; submission not written.")

    sample_path = data_dir / "sample_submission.csv"
    if sample_path.exists():
        sub = pd.read_csv(sample_path)
        sub_cols = list(sub.columns)
        if len(sub_cols) < 2:
            raise ValueError(f"Unexpected sample_submission.csv columns: {sub_cols}")
        sub_id = first_existing(sub_cols, [schema.id_col, "id", "ID", "sample_id"]) or sub_cols[0]
        target_candidates = [c for c in sub_cols if c != sub_id]
        target_col = target_candidates[0]

        pred_map = dict(zip(result_df["row_id"].astype(str), result_df[pred_col].astype(str)))
        mapped = sub[sub_id].astype(str).map(pred_map)
        if mapped.isna().any():
            # If IDs don't match but lengths do, preserve official row order and use inference order.
            if len(sub) == len(result_df):
                mapped = pd.Series(result_df[pred_col].tolist(), index=sub.index)
            else:
                miss = int(mapped.isna().sum())
                raise ValueError(f"Could not map {miss} submission IDs to predictions.")
        encoded = encode_predictions_for_sample(mapped.astype(str), sub[target_col])
        sub[target_col] = encoded.values
    else:
        sub = pd.DataFrame({schema.id_col: result_df["row_id"], "answer": result_df[pred_col]})

    sub.to_csv(out_path, index=False, encoding="utf-8-sig")
    print(f"Submission saved: {out_path}")


def main() -> None:
    args = parse_args()
    methods = {x.strip().lower() for x in args.methods.split(",") if x.strip()}
    allowed = {"generate", "letter", "text", "ensemble"}
    unknown = methods - allowed
    if unknown:
        raise ValueError(f"Unknown methods: {sorted(unknown)}; allowed={sorted(allowed)}")
    if "ensemble" in methods:
        methods |= {"letter", "text"}

    random.seed(args.seed)
    np.random.seed(args.seed)
    torch.manual_seed(args.seed)

    if args.offline:
        os.environ["HF_HUB_OFFLINE"] = "1"
        os.environ["TRANSFORMERS_OFFLINE"] = "1"

    print_environment()
    dtype = validate_cuda()

    data_dir = Path(args.data_dir).resolve()
    if not data_dir.exists():
        raise FileNotFoundError(f"data-dir not found: {data_dir}")

    if args.csv:
        csv_path = Path(args.csv).resolve()
    else:
        csv_path = data_dir / ("train.csv" if args.mode == "val" else "test.csv")
    if not csv_path.exists():
        raise FileNotFoundError(f"CSV not found: {csv_path}")

    raw_df = pd.read_csv(csv_path)
    schema = detect_schema(raw_df, args.mode)
    print("Detected schema:")
    print(json.dumps(schema.__dict__, ensure_ascii=False, indent=2))

    numeric_label_base = None
    if args.mode == "val" and schema.label_col is not None:
        numeric_label_base = infer_numeric_label_base(raw_df[schema.label_col])
        if numeric_label_base is not None:
            print(f"Detected numeric label base: {numeric_label_base} ({numeric_label_base}..{numeric_label_base+3})")

    work_df = raw_df.copy()
    if args.mode == "val":
        # Randomized subset gives a fast, reproducible local comparison.
        work_df = work_df.sample(frac=1.0, random_state=args.seed).reset_index(drop=True)
    if args.limit and args.limit > 0:
        work_df = work_df.iloc[: args.limit].copy()

    output_dir = Path(args.output_dir).resolve()
    output_dir.mkdir(parents=True, exist_ok=True)

    engine = VQAEngine(
        model_id=args.model_id,
        cache_dir=args.cache_dir,
        dtype=dtype,
        max_visual_tokens=args.max_visual_tokens,
        low_max_visual_tokens=args.low_max_visual_tokens,
        offline=args.offline,
        trust_remote_code=args.trust_remote_code,
    )

    print(f"Running {len(work_df)} rows with methods={sorted(methods)}")
    records: List[Dict[str, object]] = []
    start = time.time()

    for _, row in tqdm(work_df.iterrows(), total=len(work_df), desc="VQA"):
        options = [str(row[c]) for c in schema.option_cols]
        row_id = str(row[schema.id_col])
        image_path = resolve_image_path(data_dir, row[schema.image_col], args.mode)
        question = str(row[schema.question_col])

        rec: Dict[str, object] = {
            "row_id": row_id,
            "image_path": str(image_path),
            "question": question,
            "A": options[0],
            "B": options[1],
            "C": options[2],
            "D": options[3],
        }
        if args.mode == "val" and schema.label_col is not None:
            rec["true_label"] = normalize_label(row[schema.label_col], options, numeric_base=numeric_label_base)

        try:
            pred, used_low = engine.infer_with_oom_fallback(
                image_path=image_path,
                question=question,
                options=options,
                methods=methods,
                max_new_tokens=args.max_new_tokens,
            )
            rec.update(pred)
            rec["oom_low_res_retry"] = bool(used_low)
            rec["error"] = ""
        except Exception as e:
            rec["error"] = f"{type(e).__name__}: {e}"
            rec["oom_low_res_retry"] = False
            print(f"\n[ERROR] row_id={row_id}: {rec['error']}")

        records.append(rec)

    result_df = pd.DataFrame(records)
    elapsed = time.time() - start
    result_df["elapsed_total_sec"] = elapsed

    result_path = output_dir / f"experiment_{args.mode}.csv"
    result_df.to_csv(result_path, index=False, encoding="utf-8-sig")

    summary = compute_summary(result_df, methods)
    summary.update(
        {
            "mode": args.mode,
            "model_id": args.model_id,
            "methods": sorted(methods),
            "dtype": str(dtype),
            "elapsed_sec": elapsed,
            "rows_per_sec": (len(result_df) / elapsed) if elapsed > 0 else None,
            "errors": int((result_df.get("error", pd.Series(dtype=str)).fillna("") != "").sum()),
            "oom_low_res_retries": int(result_df.get("oom_low_res_retry", pd.Series(dtype=bool)).fillna(False).sum()),
        }
    )
    summary_path = output_dir / f"summary_{args.mode}.json"
    summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")

    print("\n" + "=" * 80)
    print("Summary")
    print(json.dumps(summary, ensure_ascii=False, indent=2))
    print(f"Detailed log : {result_path}")
    print(f"Summary JSON : {summary_path}")

    if args.mode == "test":
        submission_path = output_dir / f"submission_{args.submit_scorer}.csv"
        make_submission(
            data_dir=data_dir,
            raw_df=raw_df,
            schema=schema,
            result_df=result_df,
            scorer=args.submit_scorer,
            out_path=submission_path,
        )




## 4. 실행
위 설정값을 명령행 인자로 변환해 동일한 파이프라인을 실행합니다.

In [15]:
import sys
argv = [
    "jev_vqa_notebook",
    "--data-dir", DATA_DIR,
    "--mode", MODE,
    "--model-id", MODEL_ID,
    "--cache-dir", CACHE_DIR,
    "--output-dir", OUTPUT_DIR,
    "--methods", METHODS,
    "--submit-scorer", SUBMIT_SCORER,
    "--limit", str(LIMIT),
    "--seed", str(SEED),
    "--max-visual-tokens", str(MAX_VISUAL_TOKENS),
    "--low-max-visual-tokens", str(LOW_MAX_VISUAL_TOKENS),
    "--max-new-tokens", str(MAX_NEW_TOKENS),
]
if OFFLINE:
    argv.append("--offline")
old_argv = sys.argv
try:
    sys.argv = argv
    main()
finally:
    sys.argv = old_argv


Environment
Python       : 3.12.10
PyTorch      : 2.12.1+cu130
TorchVision  : 0.27.1+cu130
Transformers : 5.17.0
CUDA runtime : 13.0
CUDA avail   : True
GPU          : NVIDIA GeForce RTX 5060 Ti
VRAM         : 15.93 GB
Compute cap  : 12.0
BF16 support : True
nvidia-smi   : 610.88, NVIDIA GeForce RTX 5060 Ti, 16311 MiB
Detected schema:
{
  "id_col": "id",
  "image_col": "id",
  "question_col": "question",
  "option_cols": [
    "a",
    "b",
    "c",
    "d"
  ],
  "label_col": "answer"
}
Loading processor: Qwen/Qwen2.5-VL-3B-Instruct
Loading model: Qwen/Qwen2.5-VL-3B-Instruct (torch.bfloat16)


Loading weights: 100%|██████████| 824/824 [00:04<00:00, 186.42it/s]


Candidate tokenization:
  A: [32]
  B: [33]
  C: [34]
  D: [35]
Running 300 rows with methods=['ensemble', 'generate', 'letter', 'text']


VQA: 100%|██████████| 300/300 [12:34<00:00,  2.52s/it]


Summary
{
  "n": 300,
  "generate_accuracy": 0.8833333333333333,
  "generate_valid_n": 300,
  "letter_accuracy": 0.8833333333333333,
  "letter_valid_n": 300,
  "text_accuracy": 0.81,
  "text_valid_n": 300,
  "ensemble_accuracy": 0.88,
  "ensemble_valid_n": 300,
  "generate_letter_agreement": 1.0,
  "generate_wrong_letter_correct": 0,
  "generate_correct_letter_wrong": 0,
  "letter_text_agreement": 0.8433333333333334,
  "mode": "val",
  "model_id": "Qwen/Qwen2.5-VL-3B-Instruct",
  "methods": [
    "ensemble",
    "generate",
    "letter",
    "text"
  ],
  "dtype": "torch.bfloat16",
  "elapsed_sec": 754.6771728992462,
  "rows_per_sec": 0.3975209676045836,
  "errors": 0,
  "oom_low_res_retries": 0
}
Detailed log : C:\SSAFY\AIChallenge\outputs_jev\experiment_val.csv
Summary JSON : C:\SSAFY\AIChallenge\outputs_jev\summary_val.json


## 5. 결과 확인
Validation에서는 정확도와 disagreement를 확인하고, test에서는 `submission_<scorer>.csv`가 생성됩니다.

In [16]:
from pathlib import Path
import pandas as pd, json
out = Path(OUTPUT_DIR)
exp = out / f"experiment_{MODE}.csv"
summary = out / f"summary_{MODE}.json"
if exp.exists():
    display(pd.read_csv(exp).head(20))
if summary.exists():
    print(json.dumps(json.loads(summary.read_text(encoding="utf-8")), ensure_ascii=False, indent=2))
if MODE == "test":
    sub = out / f"submission_{SUBMIT_SCORER}.csv"
    print("submission:", sub.resolve() if sub.exists() else "not created")


,row_id,image_path,question,A,B,C,D,true_label,generate_pred,generate_text,...,ensemble_pred,ensemble_confidence,ensemble_margin,ensemble_p_A,ensemble_p_B,ensemble_p_C,ensemble_p_D,oom_low_res_retry,error,elapsed_total_sec
0,train_1194.jpg,C:\SSAFY\AIChallenge\dataset\train\train_1194.jpg,이 극장 간판에 적힌 문구 중 올바른 것은 무엇인가요?,YOU ALREADY KNOW YOU'RE GONNA LEAVE IT!,YOU ALREADY KNOW YOU'RE GONNA MISS IT!,YOU ALREADY KNOW YOU'RE GONNA LOVE IT!,YOU ALREADY KNOW YOU'RE GONNA HATE IT!,C,C,C,...,C,0.719882,0.612289,0.107593,0.091390,0.719882,0.081135,False,NaN,754.677173
1,train_3865.jpg,C:\SSAFY\AIChallenge\dataset\train\train_3865.jpg,이 건물에 입주해 있는 학원의 이름은 무엇인가요?,이테리어,ABC미용실,센트럴 약국,썬탑 영수 전문학원,D,D,D,...,D,0.694440,0.567621,0.063883,0.114858,0.126819,0.694440,False,NaN,754.677173
2,train_0682.jpg,C:\SSAFY\AIChallenge\dataset\train\train_0682.jpg,이 건물 2층에 있는 업체 이름은 무엇인가요?,청용 지점,신용카드 사은행 이벤트,에핏 스피닝 그룹 PT,MG 천안새마을금고,C,C,C,...,D,0.526534,0.146041,0.043359,0.049614,0.380493,0.526534,False,NaN,754.677173
3,train_3708.jpg,C:\SSAFY\AIChallenge\dataset\train\train_3708.jpg,이 가게의 이름은 무엇인가요?,G마켓,G스토어,G숍,G마트,D,D,D,...,D,0.751592,0.600043,0.151549,0.095115,0.001743,0.751592,False,NaN,754.677173
4,train_3576.jpg,C:\SSAFY\AIChallenge\dataset\train\train_3576.jpg,이 장난감 가격표에 따르면 'Woody Sheriff'의 가격은 얼마인가요?,HK$200,HK$150,HK$190,HK$179,D,D,D,...,D,0.649697,0.470879,0.096421,0.075064,0.178818,0.649697,False,NaN,754.677173
5,train_5924.jpg,C:\SSAFY\AIChallenge\dataset\train\train_5924.jpg,이미지 속 제품 포장지에 표시된 크런치의 함량은 몇 퍼센트인가요?,10%,50%,25%,75%,C,C,C,...,C,0.745694,0.655682,0.089650,0.074644,0.745694,0.090012,False,NaN,754.677173
6,train_6530.jpg,C:\SSAFY\AIChallenge\dataset\train\train_6530.jpg,이곳의 주소는 무엇입니까?,강남구 역삼동 707-4,공개공지 안내,에스원 1588-3112,테헤란로 316,D,D,D,...,D,0.662059,0.494194,0.125310,0.044766,0.167865,0.662059,False,NaN,754.677173
7,train_1050.jpg,C:\SSAFY\AIChallenge\dataset\train\train_1050.jpg,이곳에 설치된 CCTV의 주요 목적은 무엇인가요?,어린이 안전과 시설 보안,화재 예방,쓰레기 투기 감시,교통 단속을 위해,A,A,A,...,A,0.831237,0.722963,0.831237,0.031461,0.108274,0.029027,False,NaN,754.677173
8,train_6586.jpg,C:\SSAFY\AIChallenge\dataset\train\train_6586.jpg,이 사진에 적힌 전화번호는 무엇인가요?,010-1234-5678,1588-1234,02-3456-7890,1544-6764,D,D,D,...,D,0.695340,0.583668,0.111673,0.096278,0.096709,0.695340,False,NaN,754.677173
9,train_4601.jpg,C:\SSAFY\AIChallenge\dataset\train\train_4601.jpg,이 문구에서 '관계자외 출입금지'라는 의미는 무엇인가요?,누구나 자유롭게 출입할 수 있다,출입문을 항상 열어 두어야 한다,출입문 앞에 물건을 쌓아도 된다,허가받은 사람만 출입할 수 있다,D,D,D,...,D,0.679982,0.541370,0.138612,0.057299,0.124107,0.679982,False,NaN,754.677173


{
  "n": 300,
  "generate_accuracy": 0.8833333333333333,
  "generate_valid_n": 300,
  "letter_accuracy": 0.8833333333333333,
  "letter_valid_n": 300,
  "text_accuracy": 0.81,
  "text_valid_n": 300,
  "ensemble_accuracy": 0.88,
  "ensemble_valid_n": 300,
  "generate_letter_agreement": 1.0,
  "generate_wrong_letter_correct": 0,
  "generate_correct_letter_wrong": 0,
  "letter_text_agreement": 0.8433333333333334,
  "mode": "val",
  "model_id": "Qwen/Qwen2.5-VL-3B-Instruct",
  "methods": [
    "ensemble",
    "generate",
    "letter",
    "text"
  ],
  "dtype": "torch.bfloat16",
  "elapsed_sec": 754.6771728992462,
  "rows_per_sec": 0.3975209676045836,
  "errors": 0,
  "oom_low_res_retries": 0
}


## 문제 해결 메모

- `libtorchaudio.pyd` 오류: 0번 셀 실행 → **Restart Kernel** → 1번부터 실행
- `torchvision::nms` 오류: torch/torchvision CUDA wheel 조합 불일치이므로 0번 셀로 복구
- `CUDA avail: False`: 현재 VS Code Notebook 커널이 다른 Python을 가리키는지 확인
- 모델 다운로드가 끝난 뒤에는 `OFFLINE = True`로 외부 네트워크 없이 재실행 가능
